# M09a · Matrices et systèmes linéaires — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m09a_matrices/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m09a_matrices/03_notebook.ipynb`, créée par `python tools/start_chapter.py M09a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M09a.1 à M09a.14, M09a.19, M09a.20 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M09a.25 | Le produit matriciel et la transposée · `matmul, transpose` | 🔨 | ★ | 20 |
|  | M09a.26 | La forme échelonnée réduite et le rang · `rref, rank` | 🔨 | ★★ | 35 |
|  | M09a.27 | Résoudre un système par le pivot de Gauss · `solve` | 🔨 | ★★ | 30 |
|  | M09a.28 | L'inverse par Gauss-Jordan · `inverse` | 🔨 | ★ | 20 |
|  | M09a.29 | `@`, `*`, `.T`, `np.linalg.solve`, `matrix_rank` et `sympy.Matrix.rref` contre tes fonctions | 📦 | ★ | 15 |
| B | M09a.30 | Prédire la forme du résultat : `@`, `*` et broadcasting | 🔮 | ★ | 10 |
|  | M09a.31 | Le temps d'un produit matriciel en fonction de n : une pente de 3 en échelle log-log | 🔬 | ★★ | 20 |
|  | M09a.32 | Pourquoi le pivot partiel : un petit pivot et les arrondis | 🔬 | ★★ | 20 |
|  | M09a.33 | Une matrice 2 × 2 comme transformation du plan : la grille et le carré unité | 🎨 | ★ | 20 |
| C | M09a.34 | Chasse à l'erreur : `*` au lieu de `@`, une transposée oubliée, un pivot nul | 🐛 | ★ | 15 |
|  | M09a.35 | Tester une propriété : un pytest pour (AB)ᵀ = BᵀAᵀ sur des matrices aléatoires | 🛠️ | ★ | 15 |
|  | M09a.36 | Défi : un système 400 × 400 résolu par ton `solve`, à 1e−9 près de NumPy, en moins d'une seconde | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : formes et moyennes d'un tableau, produits matrice-vecteur et matriciels, coût des parenthèses, couche dense, embedding, transposées et $\mathbf{X}^\top\mathbf{X}$, pivot de Gauss, formes réduites, inverses, rangs, un problème mis en système, passe avant, standardisation.
- Écrire les six fonctions de `mymath/matrices.py` et les tester, puis comparer avec NumPy et SymPy.
- Prédire des formes, mesurer le coût $n^3$ et l'effet du pivot partiel, voir une matrice comme une transformation du plan, chasser des erreurs, tester une propriété, résoudre un grand système.

**Rappel express.** $(n, p) \times (p, m) \to (n, m)$ ; $(\mathbf{A}\mathbf{B})_{ij} = \sum_k a_{ik}b_{kj}$, $n\,p\,m$ multiplications ; $\mathbf{A}\mathbf{x} = \sum_j x_j\mathbf{a}_j$ ; $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$ ; $(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top\mathbf{A}^\top$ ; opérations élémentaires : échanger, multiplier par $\lambda \ne 0$, ajouter un multiple d'une autre ligne ; $\mathbf{A}\mathbf{A}^{-1} = \mathbf{I}$, $\begin{pmatrix} a & b \\ c & d \end{pmatrix}^{-1} = \frac{1}{ad - bc}\begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$ ; rang = nombre de pivots.

## Partie 0 · Vérifier tes exercices papier (✏️ M09a.1 à M09a.14, 🔗 M09a.19 et M09a.20)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ;
- une liste `[2, -1, 0]` pour un vecteur ou une forme, une liste de lignes `[[1, 0], [-2, 3]]` pour une matrice, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M09a.1 — Un tableau de données vu comme une matrice : forme, lignes, colonnes, moyennes**

In [ ]:
# Ex M09a.1: the values you found on paper
answer_M09a_1a = ...  # a) a list of 2 integers
answer_M09a_1b = ...  # b) a number
answer_M09a_1c = ...  # c) a list of 2 integers
answer_M09a_1d = ...  # d) a list of 3 numbers
answer_M09a_1e = ...  # e) a list of 3 numbers
answer_M09a_1f = ...  # f) a list of 3 numbers
answer_M09a_1g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M09a_1a, answer_M09a_1b, answer_M09a_1c, answer_M09a_1d, answer_M09a_1e, answer_M09a_1f, answer_M09a_1g]):
    wbm.check(f"M09a.1{letter}", answer)

**Ex M09a.2 — Le produit matrice-vecteur, par les lignes puis par les colonnes**

In [ ]:
# Ex M09a.2: the values you found on paper
answer_M09a_2a = ...  # a) an integer
answer_M09a_2b = ...  # b) a list of 3 numbers
answer_M09a_2c = ...  # c) an integer
answer_M09a_2d = ...  # d) a list of 3 numbers
answer_M09a_2e = ...  # e) a list of 4 numbers
answer_M09a_2f = ...  # f) True or False
answer_M09a_2g = ...  # g) a list of 2 numbers

for letter, answer in zip("abcdefg", [answer_M09a_2a, answer_M09a_2b, answer_M09a_2c, answer_M09a_2d, answer_M09a_2e, answer_M09a_2f, answer_M09a_2g]):
    wbm.check(f"M09a.2{letter}", answer)

**Ex M09a.3 — Produits matriciels à la main : formes compatibles et calculs**

In [ ]:
# Ex M09a.3: the values you found on paper
answer_M09a_3a = ...  # a) a list of 2 integers
answer_M09a_3b = ...  # b) a matrix
answer_M09a_3c = ...  # c) a list of 2 integers
answer_M09a_3d = ...  # d) a number
answer_M09a_3e = ...  # e) True or False
answer_M09a_3f = ...  # f) a matrix
answer_M09a_3g = ...  # g) a matrix

for letter, answer in zip("abcdefg", [answer_M09a_3a, answer_M09a_3b, answer_M09a_3c, answer_M09a_3d, answer_M09a_3e, answer_M09a_3f, answer_M09a_3g]):
    wbm.check(f"M09a.3{letter}", answer)

**Ex M09a.4 — Le coût d'un produit, et l'ordre des parenthèses dans ABC**

In [ ]:
# Ex M09a.4: the values you found on paper
answer_M09a_4a = ...  # a) an integer
answer_M09a_4b = ...  # b) an integer
answer_M09a_4c = ...  # c) an integer
answer_M09a_4d = ...  # d) a number (1 decimal)
answer_M09a_4e = ...  # e) a list of 2 integers
answer_M09a_4f = ...  # f) an integer
answer_M09a_4g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M09a_4a, answer_M09a_4b, answer_M09a_4c, answer_M09a_4d, answer_M09a_4e, answer_M09a_4f, answer_M09a_4g]):
    wbm.check(f"M09a.4{letter}", answer)

**Ex M09a.5 — Une couche dense Z = XW + b à la main : deux exemples, trois entrées, deux sorties**

In [ ]:
# Ex M09a.5: the values you found on paper
answer_M09a_5a = ...  # a) a list of 2 integers
answer_M09a_5b = ...  # b) a matrix
answer_M09a_5c = ...  # c) a number
answer_M09a_5d = ...  # d) a matrix
answer_M09a_5e = ...  # e) an integer
answer_M09a_5f = ...  # f) a list of 2 integers
answer_M09a_5g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M09a_5a, answer_M09a_5b, answer_M09a_5c, answer_M09a_5d, answer_M09a_5e, answer_M09a_5f, answer_M09a_5g]):
    wbm.check(f"M09a.5{letter}", answer)

**Ex M09a.6 — Un embedding : une ligne one-hot fois une matrice**

In [ ]:
# Ex M09a.6: the values you found on paper
answer_M09a_6a = ...  # a) a list of 5 numbers
answer_M09a_6b = ...  # b) a list of 3 numbers
answer_M09a_6c = ...  # c) an integer
answer_M09a_6d = ...  # d) an integer
answer_M09a_6e = ...  # e) a list of 3 numbers
answer_M09a_6f = ...  # f) a list of 3 numbers (4 decimals)

for letter, answer in zip("abcdef", [answer_M09a_6a, answer_M09a_6b, answer_M09a_6c, answer_M09a_6d, answer_M09a_6e, answer_M09a_6f]):
    wbm.check(f"M09a.6{letter}", answer)

**Ex M09a.7 — Transposées de produits, et la matrice symétrique XᵀX**

In [ ]:
# Ex M09a.7: the values you found on paper
answer_M09a_7a = ...  # a) a list of 2 integers
answer_M09a_7b = ...  # b) a matrix
answer_M09a_7c = ...  # c) a matrix
answer_M09a_7d = ...  # d) True or False
answer_M09a_7e = ...  # e) a matrix
answer_M09a_7f = ...  # f) a list of 2 integers
answer_M09a_7g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M09a_7a, answer_M09a_7b, answer_M09a_7c, answer_M09a_7d, answer_M09a_7e, answer_M09a_7f, answer_M09a_7g]):
    wbm.check(f"M09a.7{letter}", answer)

**Ex M09a.8 — Le pivot de Gauss sur un système 3 × 3 à solution unique**

In [ ]:
# Ex M09a.8: the values you found on paper
answer_M09a_8a = ...  # a) a number
answer_M09a_8b = ...  # b) a list of 4 numbers
answer_M09a_8c = ...  # c) a list of 4 numbers
answer_M09a_8d = ...  # d) an irreducible fraction
answer_M09a_8e = ...  # e) a number
answer_M09a_8f = ...  # f) a list of 3 numbers

for letter, answer in zip("abcdef", [answer_M09a_8a, answer_M09a_8b, answer_M09a_8c, answer_M09a_8d, answer_M09a_8e, answer_M09a_8f]):
    wbm.check(f"M09a.8{letter}", answer)

**Ex M09a.9 — Un système sans solution, un système à une infinité de solutions**

In [ ]:
# Ex M09a.9: the values you found on paper
answer_M09a_9a = ...  # a) a list of 4 numbers
answer_M09a_9b = ...  # b) an integer
answer_M09a_9c = ...  # c) a matrix
answer_M09a_9d = ...  # d) an integer
answer_M09a_9e = ...  # e) a list of 3 numbers
answer_M09a_9f = ...  # f) a list of 3 numbers
answer_M09a_9g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M09a_9a, answer_M09a_9b, answer_M09a_9c, answer_M09a_9d, answer_M09a_9e, answer_M09a_9f, answer_M09a_9g]):
    wbm.check(f"M09a.9{letter}", answer)

**Ex M09a.10 — La forme échelonnée réduite d'une matrice 3 × 4 et la solution générale**

In [ ]:
# Ex M09a.10: the values you found on paper
answer_M09a_10a = ...  # a) a matrix
answer_M09a_10b = ...  # b) a list of integers
answer_M09a_10c = ...  # c) an integer
answer_M09a_10d = ...  # d) a list of 3 numbers
answer_M09a_10e = ...  # e) a list of 3 numbers
answer_M09a_10f = ...  # f) a list of 3 numbers
answer_M09a_10g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M09a_10a, answer_M09a_10b, answer_M09a_10c, answer_M09a_10d, answer_M09a_10e, answer_M09a_10f, answer_M09a_10g]):
    wbm.check(f"M09a.10{letter}", answer)

**Ex M09a.11 — L'inverse d'une matrice 2 × 2, par la formule, puis vérifiée**

In [ ]:
# Ex M09a.11: the values you found on paper
answer_M09a_11a = ...  # a) a number
answer_M09a_11b = ...  # b) a matrix
answer_M09a_11c = ...  # c) a list of 2 numbers
answer_M09a_11d = ...  # d) a matrix
answer_M09a_11e = ...  # e) a matrix
answer_M09a_11f = ...  # f) a matrix
answer_M09a_11g = ...  # g) a matrix

for letter, answer in zip("abcdefg", [answer_M09a_11a, answer_M09a_11b, answer_M09a_11c, answer_M09a_11d, answer_M09a_11e, answer_M09a_11f, answer_M09a_11g]):
    wbm.check(f"M09a.11{letter}", answer)

**Ex M09a.12 — L'inverse d'une matrice 3 × 3 par Gauss-Jordan**

In [ ]:
# Ex M09a.12: the values you found on paper
answer_M09a_12a = ...  # a) a matrix
answer_M09a_12b = ...  # b) a list of 3 numbers
answer_M09a_12c = ...  # c) a list of 3 numbers
answer_M09a_12d = ...  # d) a list of 3 numbers
answer_M09a_12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_M09a_12a, answer_M09a_12b, answer_M09a_12c, answer_M09a_12d, answer_M09a_12e]):
    wbm.check(f"M09a.12{letter}", answer)

**Ex M09a.13 — Le rang de quatre matrices, et les colonnes redondantes**

In [ ]:
# Ex M09a.13: the values you found on paper
answer_M09a_13a = ...  # a) an integer
answer_M09a_13b = ...  # b) an integer
answer_M09a_13c = ...  # c) an integer
answer_M09a_13d = ...  # d) an integer
answer_M09a_13e = ...  # e) a list of 2 numbers
answer_M09a_13f = ...  # f) True or False
answer_M09a_13g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M09a_13a, answer_M09a_13b, answer_M09a_13c, answer_M09a_13d, answer_M09a_13e, answer_M09a_13f, answer_M09a_13g]):
    wbm.check(f"M09a.13{letter}", answer)

**Ex M09a.14 — Mettre un problème en système : trois engrais pour trois nutriments**

In [ ]:
# Ex M09a.14: the values you found on paper
answer_M09a_14a = ...  # a) a matrix
answer_M09a_14b = ...  # b) a list of 3 numbers
answer_M09a_14c = ...  # c) a number
answer_M09a_14d = ...  # d) an irreducible fraction
answer_M09a_14e = ...  # e) a list of 2 numbers

for letter, answer in zip("abcde", [answer_M09a_14a, answer_M09a_14b, answer_M09a_14c, answer_M09a_14d, answer_M09a_14e]):
    wbm.check(f"M09a.14{letter}", answer)

**Ex M09a.19 — Pont : une passe avant à deux couches, deux produits matriciels et leurs formes**

In [ ]:
# Ex M09a.19: the values you found on paper
answer_M09a_19a = ...  # a) a list of 2 integers
answer_M09a_19b = ...  # b) a list of 2 integers
answer_M09a_19c = ...  # c) an integer
answer_M09a_19d = ...  # d) an integer
answer_M09a_19e = ...  # e) a list of 3 numbers
answer_M09a_19f = ...  # f) a number
answer_M09a_19g = ...  # g) a matrix

for letter, answer in zip("abcdefg", [answer_M09a_19a, answer_M09a_19b, answer_M09a_19c, answer_M09a_19d, answer_M09a_19e, answer_M09a_19f, answer_M09a_19g]):
    wbm.check(f"M09a.19{letter}", answer)

**Ex M09a.20 — Pont : centrer et réduire un tableau en écriture matricielle**

In [ ]:
# Ex M09a.20: the values you found on paper
answer_M09a_20a = ...  # a) a list of 2 numbers
answer_M09a_20b = ...  # b) a list of 2 integers
answer_M09a_20c = ...  # c) a matrix
answer_M09a_20d = ...  # d) a list of 2 numbers (4 decimals)
answer_M09a_20e = ...  # e) a list of 2 numbers (3 decimals)
answer_M09a_20f = ...  # f) a matrix (4 decimals)
answer_M09a_20g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M09a_20a, answer_M09a_20b, answer_M09a_20c, answer_M09a_20d, answer_M09a_20e, answer_M09a_20f, answer_M09a_20g]):
    wbm.check(f"M09a.20{letter}", answer)

## Partie A · Ta librairie : produit, transposée, forme réduite, rang, systèmes, inverse ; NumPy et SymPy

Fiche §3 à §8. Les six fonctions de `mymath/matrices.py`, puis les mêmes calculs avec NumPy et SymPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"


def timed_product(A):
    """The time (seconds) of one product A @ A with NumPy."""
    start = time.perf_counter()
    A @ A
    return time.perf_counter() - start

### Ex M09a.25 — Le produit matriciel et la transposée · `matmul, transpose` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le produit matriciel et la transposée avec des opérations élément par élément, en refusant les formes incompatibles.  
**Prérequis :** fiche §3 et §4 · **mymath :** `matrices.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/matrices.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Une matrice est une liste de lignes ou un tableau NumPy à deux dimensions. Les opérations élément par élément de NumPy sont permises (`+`, `-`, `*` entre des lignes, le broadcasting, `np.sum`, `np.zeros`, `np.eye`, `np.hstack`) ; `@`, `np.dot`, `np.matmul`, `.T`, `np.transpose` et `np.linalg` sont interdits : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Ne modifie jamais la matrice reçue : travaille sur une copie, `np.array(A, dtype=float)`.

Écris (fiche §3 et §4) :
- `matmul(A, B)` : le produit $\mathbf{A}\mathbf{B}$, $(\mathbf{A}\mathbf{B})_{ij} = \sum_k a_{ik}b_{kj}$, en tableau NumPy de floats ; une `ValueError` si une des deux n'est pas à deux dimensions, ou si $\mathbf{A}$ n'a pas autant de colonnes que $\mathbf{B}$ a de lignes. Une boucle sur les éléments marche ; plus rapide : une boucle sur les lignes de $\mathbf{A}$, chaque ligne de $\mathbf{A}\mathbf{B}$ étant la combinaison des lignes de $\mathbf{B}$ pondérées par la ligne de $\mathbf{A}$ (fiche §3) ;
- `transpose(A)` : la transposée, de forme $(m, n)$ pour $\mathbf{A}$ de forme $(n, m)$, dans un **nouveau** tableau ; une `ValueError` si $\mathbf{A}$ n'est pas à deux dimensions.

La vérification calcule :
- a) `matmul([[2, -1, 0], [1, 3, 2]], [[1, 4], [0, -2], [3, 1]])` (une matrice) ;
- b) `matmul([[0.5, -2, 1]], [[4], [1], [-3]])` (une matrice) ;
- c) `transpose([[1.5, -2], [0, 4], [3, 1]])` (une matrice) ;
- d) le nom de l'erreur levée par `matmul([[1, 2, 3]], [[1, 2, 3]])` ;
- e) le nom du type de `matmul([[1]], [[2]])`, en texte ;
- f) vrai ou faux : `transpose(matmul(A, B))` est égal à `matmul(transpose(B), transpose(A))` pour les matrices de a) (calculé avec tes fonctions).

Dans ta copie : calcule a) et b) à la main. Combien de multiplications fait ta fonction pour a) ? Pourquoi `transpose` doit-elle renvoyer un nouveau tableau ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09a")  # reload your saved file

In [ ]:
A_25, B_25 = [[2, -1, 0], [1, 3, 2]], [[1, 4], [0, -2], [3, 1]]
with checked("M09a.25a"):
    wbm.check("M09a.25a", np.asarray(mymath.matrices.matmul(A_25, B_25)).tolist(), computed=True)
with checked("M09a.25b"):
    wbm.check("M09a.25b", np.asarray(mymath.matrices.matmul([[0.5, -2, 1]], [[4], [1], [-3]])).tolist(),
              computed=True)
with checked("M09a.25c"):
    wbm.check("M09a.25c", np.asarray(mymath.matrices.transpose([[1.5, -2], [0, 4], [3, 1]])).tolist(), computed=True)
with checked("M09a.25d"):
    wbm.check("M09a.25d", raised(mymath.matrices.matmul, [[1, 2, 3]], [[1, 2, 3]]), computed=True)
with checked("M09a.25e"):
    wbm.check("M09a.25e", type(mymath.matrices.matmul([[1]], [[2]])).__name__, computed=True)
with checked("M09a.25f"):
    left_25 = np.asarray(mymath.matrices.transpose(mymath.matrices.matmul(A_25, B_25)), dtype=float)
    right_25 = np.asarray(mymath.matrices.matmul(mymath.matrices.transpose(B_25), mymath.matrices.transpose(A_25)),
                          dtype=float)
    wbm.check("M09a.25f", bool(left_25.shape == right_25.shape and np.allclose(left_25, right_25)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09a_matrices.py", "test_matmul_ or test_transpose_ or test_matrices_uses_")

### Ex M09a.26 — La forme échelonnée réduite et le rang · `rref, rank` 🔨 ★★ ⏱️ 35 min
**Objectif :** programmer l'élimination de Gauss-Jordan avec pivot partiel, et en déduire le rang.  
**Prérequis :** fiche §6 et §8 · **mymath :** `matrices.py` · **Parcours :** C, E

Écris (fiche §6 et §8), en suivant la docstring :
- `rref(A, tol=1e-10)` : la forme échelonnée réduite et le tuple des colonnes des pivots. Colonne par colonne depuis la gauche, avec une ligne courante $r$ (au début 0) : dans la colonne courante, prends la ligne (à partir de $r$) dont l'élément a la plus grande valeur absolue ; s'il vaut au plus `tol`, la colonne n'a pas de pivot ; sinon échange cette ligne avec la ligne $r$, divise-la par le pivot, retranche-en des multiples à **toutes** les autres lignes, note la colonne et passe à la ligne $r + 1$. À la fin, mets à 0 les nombres au plus égaux à `tol` en valeur absolue ;
- `rank(A, tol=1e-10)` : le nombre de pivots, en `int` de Python.

Les deux lèvent une `ValueError` si $\mathbf{A}$ n'est pas à deux dimensions. La forme réduite est unique : les tests la comparent à celle de SymPy.

La vérification calcule :
- a) la forme réduite de `rref([[1, 2, -1, 3], [2, 4, 1, 0], [-1, -2, 2, -3]])` (une matrice) ;
- b) ses colonnes des pivots (une liste) ;
- c) la forme réduite de `rref([[0, 0, 2], [0, 3, 1], [0, 6, 4]])` (une matrice) ;
- d) `rank([[1, -1, 2, 0], [2, -2, 4, 0], [0, 1, 1, 1], [1, 0, 3, 1]])` ;
- e) le nom de l'erreur levée par `rank([1, 2, 3])` ;
- f) le nom du type de la valeur rendue par `rank`, en texte.

Dans ta copie : pourquoi la deuxième colonne (indice 1) n'a-t-elle pas de pivot en a) ? Quelles lignes de d) sont des combinaisons des autres ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09a")  # reload your saved file

In [ ]:
with checked("M09a.26a"):
    wbm.check("M09a.26a", np.asarray(mymath.matrices.rref([[1, 2, -1, 3], [2, 4, 1, 0], [-1, -2, 2, -3]])[0]).tolist(),
              computed=True)
with checked("M09a.26b"):
    wbm.check("M09a.26b", list(mymath.matrices.rref([[1, 2, -1, 3], [2, 4, 1, 0], [-1, -2, 2, -3]])[1]),
              computed=True)
with checked("M09a.26c"):
    wbm.check("M09a.26c", np.asarray(mymath.matrices.rref([[0, 0, 2], [0, 3, 1], [0, 6, 4]])[0]).tolist(),
              computed=True)
with checked("M09a.26d"):
    wbm.check("M09a.26d", mymath.matrices.rank([[1, -1, 2, 0], [2, -2, 4, 0], [0, 1, 1, 1], [1, 0, 3, 1]]),
              computed=True)
with checked("M09a.26e"):
    wbm.check("M09a.26e", raised(mymath.matrices.rank, [1, 2, 3]), computed=True)
with checked("M09a.26f"):
    wbm.check("M09a.26f", type(mymath.matrices.rank([[1, 3], [2, 5]])).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09a_matrices.py", "test_rref_ or test_rank_ or test_matrices_uses_")

### Ex M09a.27 — Résoudre un système par le pivot de Gauss · `solve` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer la résolution d'un système carré par élimination avec pivot partiel puis remontée, et refuser une matrice singulière.  
**Prérequis :** fiche §5 et §6 · **mymath :** `matrices.py` · **Parcours :** C, E

Écris `solve(A, b)` (fiche §6), en suivant la docstring : sur la matrice augmentée $[\mathbf{A} \mid \mathbf{b}]$ (`np.hstack`), pour chaque colonne $k$, prends pour pivot la ligne (à partir de $k$) dont l'élément a la plus grande valeur absolue, échange-la avec la ligne $k$, et annule les éléments **sous** le pivot ; puis remonte, de la dernière ligne à la première. Une `ValueError` si $\mathbf{A}$ n'est pas carrée, si $\mathbf{b}$ n'a pas $n$ coordonnées, ou si un pivot vaut au plus $10^{-12}$ fois la plus grande valeur absolue de $\mathbf{A}$ (matrice singulière).

La vérification calcule :
- a) `solve([[3, -1, 2], [1, 2, -1], [2, 1, 1]], [1, 5, 4])` (une liste, 6 décimales) ;
- b) `solve([[0, 2, 1], [1, 1, 1], [2, 0, 3]], [5, 6, 13])` (un pivot nul au départ ; une liste, 6 décimales) ;
- c) `solve([[1e-18, 1], [1, 2]], [1, 3])` (une liste, 6 décimales) ;
- d) le nom de l'erreur levée par `solve([[1, 2, 3], [2, 4, 6], [1, 0, 1]], [1, 2, 3])` ;
- e) le nom de l'erreur levée par `solve([[1, 2], [3, 5]], [1, 2, 3])` ;
- f) le nom du type de la solution rendue, en texte.

Dans ta copie : vérifie a) dans les trois équations. Que vaudrait c) sans pivot partiel ? Pourquoi d) est-elle singulière ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09a")  # reload your saved file

In [ ]:
with checked("M09a.27a"):
    wbm.check("M09a.27a", np.asarray(mymath.matrices.solve([[3, -1, 2], [1, 2, -1], [2, 1, 1]], [1, 5, 4])).tolist(),
              computed=True)
with checked("M09a.27b"):
    wbm.check("M09a.27b", np.asarray(mymath.matrices.solve([[0, 2, 1], [1, 1, 1], [2, 0, 3]], [5, 6, 13])).tolist(),
              computed=True)
with checked("M09a.27c"):
    wbm.check("M09a.27c", np.asarray(mymath.matrices.solve([[1e-18, 1], [1, 2]], [1, 3])).tolist(), computed=True)
with checked("M09a.27d"):
    wbm.check("M09a.27d", raised(mymath.matrices.solve, [[1, 2, 3], [2, 4, 6], [1, 0, 1]], [1, 2, 3]), computed=True)
with checked("M09a.27e"):
    wbm.check("M09a.27e", raised(mymath.matrices.solve, [[1, 2], [3, 5]], [1, 2, 3]), computed=True)
with checked("M09a.27f"):
    wbm.check("M09a.27f", type(mymath.matrices.solve([[2, 1], [1, 1]], [3, 2])).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09a_matrices.py", "test_solve_ or test_matrices_uses_")

### Ex M09a.28 — L'inverse par Gauss-Jordan · `inverse` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer l'inverse d'une matrice carrée en réduisant $[\mathbf{A} \mid \mathbf{I}]$, et refuser une matrice singulière.  
**Prérequis :** fiche §7 ; Ex M09a.26 · **mymath :** `matrices.py` · **Parcours :** C

Écris `inverse(A)` (fiche §7) : construis $[\mathbf{A} \mid \mathbf{I}_n]$ (`np.hstack` et `np.eye` sont permis), réduis-la avec **ta** fonction `rref` (écris d'abord M09a.26), vérifie que la partie gauche est l'identité (les $n$ premiers pivots sont dans les colonnes $0, \ldots, n - 1$), et rends la partie droite. Une `ValueError` si $\mathbf{A}$ n'est pas carrée ou est singulière.

La vérification calcule :
- a) `inverse([[2, 0, 1], [1, 1, 0], [0, 2, 1]])` (une matrice, 6 décimales) ;
- b) `inverse([[0, 0, 1], [1, 0, 0], [0, 1, 0]])` (une matrice) ;
- c) le nom de l'erreur levée par `inverse([[1, -1, 2], [0, 1, 1], [1, 0, 3]])` ;
- d) le nom de l'erreur levée par `inverse([[1, 0, 2], [3, 1, 1]])` ;
- e) vrai ou faux : `matmul(A, inverse(A))` est l'identité, à $10^{-12}$ près, pour la matrice de a) (avec tes fonctions).

Dans ta copie : que remarques-tu en comparant b) à la transposée de la matrice ? Pourquoi c) est-elle singulière ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M09a")  # reload your saved file

In [ ]:
A_28 = [[2, 0, 1], [1, 1, 0], [0, 2, 1]]
with checked("M09a.28a"):
    wbm.check("M09a.28a", np.asarray(mymath.matrices.inverse(A_28)).tolist(), computed=True)
with checked("M09a.28b"):
    wbm.check("M09a.28b", np.asarray(mymath.matrices.inverse([[0, 0, 1], [1, 0, 0], [0, 1, 0]])).tolist(),
              computed=True)
with checked("M09a.28c"):
    wbm.check("M09a.28c", raised(mymath.matrices.inverse, [[1, -1, 2], [0, 1, 1], [1, 0, 3]]), computed=True)
with checked("M09a.28d"):
    wbm.check("M09a.28d", raised(mymath.matrices.inverse, [[1, 0, 2], [3, 1, 1]]), computed=True)
with checked("M09a.28e"):
    product_28 = np.asarray(mymath.matrices.matmul(A_28, mymath.matrices.inverse(A_28)), dtype=float)
    wbm.check("M09a.28e", bool(np.allclose(product_28, np.eye(3), rtol=0, atol=1e-12)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m09a_matrices.py", "test_inverse_ or test_matrices_uses_")

### Ex M09a.29 — `@`, `*`, `.T`, `np.linalg.solve`, `matrix_rank` et `sympy.Matrix.rref` contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** faire les mêmes calculs avec NumPy et SymPy, lire ce que rend chaque fonction, et comparer avec ta librairie.  
**Prérequis :** fiche §3, §6 et §8 ; Ex M09a.25 à M09a.27 · **Parcours :** C, E

En pratique, on ne réécrit pas ces fonctions :
- `A @ B` est le produit matriciel, `A * B` le produit de Hadamard (élément par élément, avec broadcasting), `A.T` la transposée ;
- `np.linalg.solve(A, b)` résout un système carré, `np.linalg.matrix_rank(A)` donne le rang (par une autre méthode que le pivot, la SVD de M12d), `np.linalg.inv(A)` l'inverse ;
- `sympy.Matrix(A).rref()` rend la forme échelonnée réduite **exacte** (avec des fractions) et le tuple des colonnes des pivots.

Avec $\mathbf{A}$ = `[[1, -2, 0, 3], [2, 1, -1, 0]]`, $\mathbf{B}$ = `[[0, 1], [2, -1], [1, 1], [-1, 2]]` et $\mathbf{P}$ = `[[2, 4, 1, 3], [1, 2, 0, 1], [3, 6, 1, 4]]`, calcule :
- a) `A @ B` (une matrice) ;
- b) `np.array([[2, -1], [3, 0]]) * np.array([[0, 1], [-1, 2]])` (une matrice) ;
- c) `np.linalg.matrix_rank(P)` ;
- d) les colonnes des pivots de `sympy.Matrix(P).rref()` (une liste) ;
- e) `np.linalg.solve([[4, 1], [2, 3]], [9, 7])` (une liste, 6 décimales) ;
- f) vrai ou faux : pour 20 systèmes $5 \times 5$ tirés au hasard (`np.random.default_rng(29)`), ta fonction `solve` donne la même solution que `np.linalg.solve` à $10^{-9}$ près (calcule-le avec une boucle).

Dans ta copie : calcule à la main l'élément $(1, 2)$ de a). Que donnerait `A * B`, et pourquoi ?

In [ ]:
answer_M09a_29a = ...  # a) a matrix
answer_M09a_29b = ...  # b) a matrix
answer_M09a_29c = ...  # c) an integer
answer_M09a_29d = ...  # d) a list
answer_M09a_29e = ...  # e) a list (6 decimals)
answer_M09a_29f = ...  # f) True or False, computed with a loop

In [ ]:
for letter, answer in zip("abcdef", [answer_M09a_29a, answer_M09a_29b, answer_M09a_29c, answer_M09a_29d,
                                     answer_M09a_29e, answer_M09a_29f]):
    wbm.check(f"M09a.29{letter}", answer)

## Partie B · Une prédiction, deux expériences et une figure

Fiche §1, §2, §3 et §6. Les formes de `@` et de `*`, le temps d'un produit, le pivot partiel, une matrice vue comme une transformation du plan.

### Ex M09a.30 — Prédire la forme du résultat : `@`, `*` et broadcasting 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la forme du résultat de `@` et de `*` sur des tableaux de formes différentes, puis la vérifier.  
**Prérequis :** fiche §1 et §3 · **Parcours :** C, E

NumPy applique deux règles différentes. Pour `@`, celle du produit matriciel (un tableau à une dimension y joue le rôle d'un vecteur). Pour `*`, le **broadcasting** : les formes se comparent en partant de la droite, et une dimension égale à 1 est répétée autant qu'il faut ; si deux dimensions diffèrent et qu'aucune ne vaut 1, c'est une erreur. Choisis, pour chaque expression, le numéro de la bonne réponse :
- a) `np.ones((2, 3)) @ np.ones((3, 1))` : (1) la forme `(2,)` ; (2) une erreur ; (3) la forme `(2, 1)` ;
- b) `np.ones((3, 4)) @ np.ones(4)` : (1) la forme `(3,)` ; (2) une erreur ; (3) la forme `(3, 4)` ;
- c) `np.ones(4) @ np.ones((4, 2))` : (1) une erreur ; (2) la forme `(4, 2)` ; (3) la forme `(2,)` ;
- d) `np.ones((2, 3)) * np.ones(3)` : (1) une erreur ; (2) la forme `(2, 3)` ; (3) la forme `(3,)` ;
- e) `np.ones((2, 3)) * np.ones(2)` : (1) la forme `(2, 3)` ; (2) la forme `(2,)` ; (3) une erreur.

**Avant d'exécuter quoi que ce soit**, écris tes cinq prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M09a_30a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M09a_30b = ...  # b) 1, 2 or 3
prediction_M09a_30c = ...  # c) 1, 2 or 3
prediction_M09a_30d = ...  # d) 1, 2 or 3
prediction_M09a_30e = ...  # e) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcde", [prediction_M09a_30a, prediction_M09a_30b, prediction_M09a_30c,
                                       prediction_M09a_30d, prediction_M09a_30e]):
    wbm.check(f"M09a.30{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M09a_30a, prediction_M09a_30b, prediction_M09a_30c, prediction_M09a_30d, prediction_M09a_30e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    for label, make in [("a) ones((2, 3)) @ ones((3, 1))", lambda: np.ones((2, 3)) @ np.ones((3, 1))),
                        ("b) ones((3, 4)) @ ones(4)", lambda: np.ones((3, 4)) @ np.ones(4)),
                        ("c) ones(4) @ ones((4, 2))", lambda: np.ones(4) @ np.ones((4, 2))),
                        ("d) ones((2, 3)) * ones(3)", lambda: np.ones((2, 3)) * np.ones(3)),
                        ("e) ones((2, 3)) * ones(2)", lambda: np.ones((2, 3)) * np.ones(2))]:
        try:
            print(f"{label:32s} → shape {make().shape}")
        except ValueError as exc:
            print(f"{label:32s} → ValueError: {str(exc)[:60]}")

**Après l'expérience**, dans ta copie : pourquoi a) rend-il un tableau à deux dimensions, et b) un tableau à une seule ? Pourquoi d) marche-t-il et pas e) ? Comment écrire, pour e), un tableau de forme `(2, 1)` qui ajoute un nombre à chaque ligne ?

### Ex M09a.31 — Le temps d'un produit matriciel en fonction de n : une pente de 3 en échelle log-log 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le temps d'un produit matriciel naïf en fonction de la taille, et lire l'exposant $3$ sur une droite en échelle log-log.  
**Prérequis :** fiche §3 ; R6 (logarithmes) · **Parcours :** C

Un produit de deux matrices carrées de taille $n$ coûte $n^3$ multiplications (fiche §3) : son temps devrait varier comme $n^3$, et, en échelle log-log, $\ln t \approx 3\ln n + c$, une droite de pente 3. Écris :
- `naive_matmul(A, B)` : le produit de deux listes de listes par une **triple boucle** Python (sans NumPy), qui rend une liste de listes ;
- `time_once(n, seed=0)` : le temps (en secondes, `time.perf_counter()`) de `naive_matmul(A, A)` pour une matrice $n \times n$ `A = np.random.default_rng(seed).normal(size=(n, n)).tolist()` ;
- `loglog_slope(ns, times)` : la pente de la droite qui ajuste $\ln t$ en fonction de $\ln n$ (`np.polyfit(..., 1)[0]`).

La vérification mesure trois fois le temps pour chaque $n$ = 40, 80 et 160, et garde le plus petit (un temps mesuré varie d'une exécution à l'autre) ; puis elle vérifie : a) que `naive_matmul` donne le même résultat que `@` sur une petite matrice ; b) que la pente mesurée est entre 2,3 et 3,5. Une dernière cellule compare le temps de ta triple boucle à celui de `@` pour $n = 120$.

Dans ta copie : quelle pente as-tu mesurée ? Pourquoi n'est-elle pas exactement 3 (pense au temps qui ne dépend pas de $n^3$) ? Pourquoi garder le plus petit de trois temps plutôt que leur moyenne ? Combien de fois `@` est-il plus rapide que ta triple boucle ?

In [ ]:
def naive_matmul(A, B):
    """The product of two lists of lists, by a triple Python loop (no NumPy), as a list of lists."""
    raise NotImplementedError("trois boucles imbriquées : lignes de A, colonnes de B, puis la somme sur k")


def time_once(n, seed=0):
    """The time (seconds) of naive_matmul(A, A) for an n × n matrix A drawn with np.random.default_rng(seed)."""
    raise NotImplementedError("time.perf_counter() avant et après le calcul")


def loglog_slope(ns, times):
    """The slope of the line that fits ln(time) as a function of ln(n)."""
    raise NotImplementedError("np.polyfit(np.log(ns), np.log(times), 1)[0]")

In [ ]:
ok_product_31, slope_31 = False, float("nan")
with checked("M09a.31 vérifications"):
    small_31 = [[1.0, -2.0, 0.5], [3.0, 0.0, -1.0]]
    other_31 = [[2.0, 1.0], [0.0, -1.0], [4.0, 3.0]]
    got_31 = naive_matmul(small_31, other_31)
    try:
        ok_product_31 = bool(np.allclose(np.array(got_31, dtype=float), np.array(small_31) @ np.array(other_31)))
    except (TypeError, ValueError):
        ok_product_31 = False
    times_31 = [min(time_once(n) for _ in range(3)) for n in (40, 80, 160)]   # the best of 3 runs
    slope_31 = float(loglog_slope((40, 80, 160), times_31))
    print("temps (s) :", [round(t, 4) for t in times_31], "· pente :", round(slope_31, 2))
    if not ok_product_31:
        print("❌ Ex M09a.31 : naive_matmul ne donne pas le produit matriciel (compare avec @ sur une petite matrice).")
    elif not 2.3 <= slope_31 <= 3.5:
        print(f"❌ Ex M09a.31 : la pente mesurée, {slope_31:.2f}, n'est pas entre 2,3 et 3,5 : vérifie time_once (le "
              "temps du seul produit) et loglog_slope (logarithmes des deux listes).")
    else:
        print("✅ Ex M09a.31 : le temps varie à peu près comme n³.")
with checked("M09a.31"):
    naive_matmul([[1.0]], [[1.0]])                     # ⏳ while naive_matmul is not written
    wbm.check("M09a.31", bool(ok_product_31 and 2.3 <= slope_31 <= 3.5), computed=True)

**La comparaison avec `@`** (cette cellule utilise ta fonction).

In [ ]:
with checked("M09a.31 comparaison"):
    A_big = np.random.default_rng(1).normal(size=(120, 120))
    start = time.perf_counter()
    naive_matmul(A_big.tolist(), A_big.tolist())
    loop_seconds = time.perf_counter() - start
    numpy_seconds = min(timed_product(A_big) for _ in range(5))   # the best of 5 runs
    print(f"triple loop: {loop_seconds:.3f} s · @: {numpy_seconds:.6f} s · ratio ≈ {loop_seconds / max(numpy_seconds, 1e-9):.0f}")

### Ex M09a.32 — Pourquoi le pivot partiel : un petit pivot et les arrondis 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'erreur d'une élimination de Gauss sans échange de lignes quand le pivot devient minuscule, et la comparer au pivot partiel.  
**Prérequis :** fiche §6 ; R1 (arrondis) ; Ex M09a.27 · **Parcours :** C

Le système $\varepsilon x + y = 1$, $x + y = 2$ a, pour tout petit $\varepsilon > 0$, une solution proche de $(1, 1)$. La cellule fournie définit `solve_no_pivot(A, b)`, une élimination de Gauss **sans** échange de lignes. Écris `errors(k_values)`, qui rend, pour chaque $k$ de la liste, le couple `(erreur sans pivot, erreur avec pivot)` pour $\varepsilon = 10^{-k}$ : l'erreur est le plus grand écart, en valeur absolue, entre la solution calculée et celle de `np.linalg.solve` ; « avec pivot » utilise ta fonction `mymath.matrices.solve` (M09a.27).

La vérification calcule :
- a) l'erreur sans pivot pour $k = 18$ (4 décimales) ;
- b) l'erreur sans pivot pour $k = 8$, par son ordre de grandeur : la vérification compare l'exposant, `round(math.log10(erreur))` ;
- c) le plus petit $k$ parmi 2, 4, …, 18 pour lequel l'erreur sans pivot dépasse $10^{-2}$ (0 s'il n'y en a aucun) ;
- d) vrai ou faux : avec le pivot partiel, l'erreur reste sous $10^{-14}$ pour tous ces $k$.

Dans ta copie : que vaut le multiplicateur $\frac{1}{\varepsilon}$ de l'élimination sans pivot, et que fait-il à la seconde équation ? Pourquoi échanger les lignes règle-t-il le problème ? Pourquoi l'erreur croît-elle à peu près comme $\varepsilon^{-1} \times 10^{-16}$ ?

In [ ]:
def solve_no_pivot(A, b):
    """Gaussian elimination WITHOUT row swaps, then back substitution (the pivots are taken on the diagonal)."""
    M = np.hstack([np.array(A, dtype=float), np.array(b, dtype=float)[:, None]])
    n = len(b)
    for k in range(n):
        for i in range(k + 1, n):
            M[i] = M[i] - M[i, k] / M[k, k] * M[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, n] - np.sum(M[i, i + 1:n] * x[i + 1:])) / M[i, i]
    return x

In [ ]:
def errors(k_values):
    """For each k, (max error without pivoting, max error with partial pivoting) for epsilon = 10**(-k)."""
    raise NotImplementedError("pour chaque k : A, b, la solution de np.linalg.solve, puis les deux écarts")

In [ ]:
K_32 = range(2, 19, 2)
with checked("M09a.32a"):
    wbm.check("M09a.32a", errors([18])[0][0], computed=True)
with checked("M09a.32b"):
    wbm.check("M09a.32b", round(math.log10(max(errors([8])[0][0], 1e-300))), computed=True)
with checked("M09a.32c"):
    wbm.check("M09a.32c", next((k for k, (e_no, _) in zip(K_32, errors(K_32)) if e_no > 1e-2), 0), computed=True)
with checked("M09a.32d"):
    wbm.check("M09a.32d", bool(all(e_yes < 1e-14 for _, e_yes in errors(K_32))), computed=True)

### Ex M09a.33 — Une matrice 2 × 2 comme transformation du plan : la grille et le carré unité 🎨 ★ ⏱️ 20 min
**Objectif :** tracer l'image de la grille et du carré unité par une matrice $2 \times 2$, et voir les colonnes de la matrice comme les images des vecteurs de base.  
**Prérequis :** fiche §2 et §3 ; R0 (matplotlib) · **Parcours :** C

Une matrice $\mathbf{M}$ de forme $(2, 2)$ envoie chaque point $\mathbf{x}$ du plan sur $\mathbf{M}\mathbf{x}$. Par la lecture en colonnes (fiche §2), $\mathbf{M}\mathbf{e}_1$ et $\mathbf{M}\mathbf{e}_2$ sont les deux colonnes de $\mathbf{M}$ : elles disent où vont les vecteurs de base, et donc tout le reste. Écris `draw_transform(ax, M)`, qui trace dans l'axe `ax` :
- l'image par $\mathbf{M}$ des lignes de la grille $x = -2, -1, 0, 1, 2$ et $y = -2, -1, 0, 1, 2$ (pour $x$ ou $y$ entre $-2$ et $2$), en gris ;
- l'image du carré unité, de sommets $(0, 0)$, $(1, 0)$, $(1, 1)$, $(0, 1)$ **dans cet ordre**, avec `matplotlib.patches.Polygon` et `ax.add_patch`, en couleur.

L'image d'une ligne est encore une ligne : il suffit de transformer ses deux extrémités (ou quelques points). La cellule de la figure s'en sert pour quatre matrices : une rotation d'un quart de tour, un cisaillement, une dilatation, et une matrice singulière. Une cellule de vérification essaie ta fonction sur les mêmes matrices, dans des axes neufs.

Dans ta copie : où vont $\mathbf{e}_1$ et $\mathbf{e}_2$ pour chaque matrice ? Que devient le carré unité pour la matrice singulière, et pourquoi (pense à ses colonnes) ? Quelle matrice ramènerait la figure de la rotation à la grille de départ ?

In [ ]:
from matplotlib.patches import Polygon


def draw_transform(ax, M):
    """In ax, the image by M of the grid lines x, y in {-2, ..., 2} (grey) and of the unit square (a Polygon)."""
    raise NotImplementedError("les extrémités de chaque ligne, transformées par M, puis ax.plot ; le carré : Polygon")

In [ ]:
with checked("M09a.33 figure"):
    matrices_33 = [("rotation", [[0, -1], [1, 0]]), ("cisaillement", [[1, 1.5], [0, 1]]),
                   ("dilatation", [[2, 0], [0, 0.5]]), ("singulière", [[1, 2], [0.5, 1]])]
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
    try:
        for ax, (name, M) in zip(axes, matrices_33):
            created = drawn(draw_transform, ax, M)
            ax.set(title=f"{name} : M = {M}", aspect="equal", xlim=(-4, 4), ylim=(-4, 4))
    except BaseException:
        plt.close(fig)
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M09a.33 : draw_transform ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur les mêmes matrices, dans des axes neufs).

In [ ]:
with checked("M09a.33 vérifications"):
    from matplotlib.patches import Polygon
    problems = []
    for M_t in ([[0, -1], [1, 0]], [[1, 1.5], [0, 1]], [[2, 0], [0, 0.5]], [[1, 2], [0.5, 1]]):
        fig_t, ax_t = plt.subplots()
        try:
            created = drawn(draw_transform, ax_t, M_t)
        except NotImplementedError:
            plt.close(fig_t)
            raise
        polygons = [patch for patch in ax_t.patches if isinstance(patch, Polygon)]
        lines_t = [line for line in ax_t.get_lines() if len(line.get_xdata()) > 1]
        plt.close(fig_t)
        if created:
            problems.append("draw_transform ouvre sa propre figure : dessine dans l'axe reçu")
            break
        if len(lines_t) < 10:
            problems.append(f"pour M = {M_t}, {len(lines_t)} lignes seulement : trace l'image des lignes de la grille "
                            "(au moins 5 verticales et 5 horizontales)")
            break
        if not polygons:
            problems.append("aucun polygone : trace l'image du carré unité avec matplotlib.patches.Polygon")
            break
        corners = np.asarray(polygons[0].get_xy(), dtype=float)[:4]
        expected = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float) @ np.array(M_t, dtype=float).T
        if corners.shape != (4, 2) or not np.allclose(corners, expected, atol=1e-9):
            problems.append(f"pour M = {M_t}, les sommets du polygone ne sont pas les images de (0, 0), (1, 0), "
                            "(1, 1), (0, 1), dans cet ordre (l'image d'un point x est M x)")
            break
        image_lines = np.concatenate([np.column_stack([line.get_xdata(), line.get_ydata()]) for line in lines_t])
        if not np.all(np.isfinite(image_lines)):
            problems.append("des points non finis dans les lignes tracées")
            break
    if problems:
        print("❌ Ex M09a.33 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M09a.33 : l'image de la grille et du carré unité est juste, pour quatre matrices.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §3, §4 et §6. Trois erreurs à trouver, un test de propriété, puis un grand système résolu vite et bien.

### Ex M09a.34 — Chasse à l'erreur : `*` au lieu de `@`, une transposée oubliée, un pivot nul 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur les produits, les formes et le pivot de Gauss, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §3, §4 et §6 · **Parcours :** C

Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle doit calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
A = np.array([[1.0, 2.0], [3.0, -1.0]])
B = np.array([[0.0, 1.0], [2.0, 2.0]])
X = np.array([[1.0, 0.0, 2.0], [-1.0, 1.0, 1.0]])      # 2 examples, 3 features
w = np.array([1.0, -2.0, 0.5])                           # weights of a linear model
P = A * B                                    # (A) the matrix product AB: its entry (2, 1), P[1, 0]
S = A + A.T                                  # (B) a symmetric matrix: S[0, 1]
y = X @ w - 0.5                              # (C) the predictions X w − 0.5 for the 2 examples: y[1]
G = X @ X.T                                  # (D) the 3 × 3 matrix XᵀX (one row per feature): G[0, 1]
M = np.array([[0.0, 2.0, 4.0], [1.0, 1.0, 3.0]])         # the augmented matrix of 2y = 4, x + y = 3
M[1] = M[1] - M[1, 0] / M[0, 0] * M[0]       # (E) an echelon form of M, x eliminated below the first pivot: M[1, 2]
R = np.diag(1 / np.array([2.0, 4.0])) @ A    # (F) D⁻¹A with D = diag(2, 4): row 1 halved, row 2 quartered: R[1, 0]
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 4 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? La ligne (D) s'exécute sans erreur : quelle est la forme de son résultat, et pourquoi Python ne signale-t-il rien ?

In [ ]:
answer_M09a_34a = ...  # a) letters, as text
answer_M09a_34b = ...  # b) a list of 6 values (4 decimals)

In [ ]:
wbm.check("M09a.34a", answer_M09a_34a)
wbm.check("M09a.34b", answer_M09a_34b)

### Ex M09a.35 — Tester une propriété : un pytest pour (AB)ᵀ = BᵀAᵀ sur des matrices aléatoires 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test fondé sur une propriété, vérifiée sur des matrices aléatoires de formes variées, et voir ce qu'un tel test ne peut pas attraper.  
**Prérequis :** fiche §4 ; Ex M09a.15 et M09a.25 · **Parcours :** C

Un test peut comparer une fonction à un **oracle** (une fonction de confiance, comme `@` dans les tests de `mymath`), ou vérifier une **propriété** que le résultat doit avoir : ici, $(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top\mathbf{A}^\top$ (M09a.15), pour des matrices tirées au hasard. Écris :
- `check_transpose_of_product(matmul, transpose, seed, trials=10)` : avec `rng = np.random.default_rng(seed)`, répète `trials` fois : tire trois dimensions $n, p, m$ entre 1 et 5 (`rng.integers(1, 6, size=3)`), deux matrices $\mathbf{A}$ $(n, p)$ et $\mathbf{B}$ $(p, m)$ de loi normale, et vérifie par `assert` (avec un message) que `transpose(matmul(A, B))` et `matmul(transpose(B), transpose(A))` ont la même forme et sont égales à $10^{-12}$ près (`np.allclose`) ;
- `test_transpose_of_product()` : le test au format pytest, sans argument, qui appelle `check_transpose_of_product` avec les fonctions de **ta** librairie (`mymath.matrices.matmul` et `mymath.matrices.transpose`) et une graine fixe.

La cellule fournie définit deux copies fausses : une transposée par `reshape`, et un produit qui rend le produit de Hadamard quand les deux matrices ont la même forme. La vérification lance ton test sur les bonnes fonctions (la référence) et sur chaque copie fausse, pour 20 graines.

Dans ta copie : pour quelles formes la transposée par `reshape` rend-elle le bon résultat, et pourquoi un test qui ne tirerait que ces formes la laisserait-il passer ? La copie au produit de Hadamard passe ton test : pourquoi la propriété est-elle vraie pour elle aussi ? Que faut-il ajouter pour l'attraper ?

In [ ]:
def transpose_reshape(A):          # a copy with a bug
    A = np.asarray(A, dtype=float)
    return A.reshape(A.shape[1], A.shape[0])


def matmul_square_hadamard(A, B):    # a copy with another bug
    A, B = np.asarray(A, dtype=float), np.asarray(B, dtype=float)
    if A.shape == B.shape:
        return A * B
    return A @ B

In [ ]:
def check_transpose_of_product(matmul, transpose, seed, trials=10):
    """Assert (AB)ᵀ = BᵀAᵀ on `trials` pairs of random matrices of random shapes, drawn with default_rng(seed)."""
    raise NotImplementedError("tire n, p, m, puis A et B, puis assert np.allclose(...), message")


def test_transpose_of_product():
    """The pytest test: check_transpose_of_product on your matmul and transpose, with a fixed seed."""
    raise NotImplementedError("un appel à check_transpose_of_product")

In [ ]:
with checked("M09a.35 vérifications"):
    problems = []
    ref_35 = wbm.load_mymath("ref").matrices

    def outcome(matmul_t, transpose_t, seed):
        """None if check_transpose_of_product passes, else the message of its AssertionError."""
        try:
            check_transpose_of_product(matmul_t, transpose_t, seed)
        except AssertionError as exc:
            return str(exc)
        return None

    passed = sum(outcome(ref_35.matmul, ref_35.transpose, seed) is None for seed in range(20))
    caught_reshape = [outcome(ref_35.matmul, transpose_reshape, seed) for seed in range(20)]
    caught_hadamard = [outcome(matmul_square_hadamard, ref_35.transpose, seed) for seed in range(20)]
    print(f"la bonne paire passe ton test {passed} fois sur 20 ; la transposée par reshape est attrapée "
          f"{sum(m is not None for m in caught_reshape)} fois sur 20 ; le produit de Hadamard pour des matrices carrées, "
          f"{sum(m is not None for m in caught_hadamard)} fois sur 20")
    if passed < 20:
        problems.append("ton test échoue sur les bonnes fonctions : compare avec np.allclose (des flottants), pas avec ==")
    if sum(m is not None for m in caught_reshape) < 20:
        problems.append("ton test laisse passer la transposée par reshape : tire des matrices de formes aléatoires, "
                        "avec au moins deux lignes et deux colonnes")
    if not problems and any(not (m or "").strip() for m in caught_reshape):
        problems.append("ajoute un message à ton assert (assert condition, message)")
    if not problems:
        try:
            test_transpose_of_product()
        except AssertionError as exc:
            problems.append(f"test_transpose_of_product échoue sur ta librairie : {exc}")
    if problems:
        print("❌ Ex M09a.35 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M09a.35 : ton test passe sur les bonnes fonctions et attrape la transposée fausse.")

### Ex M09a.36 — Défi : un système 400 × 400 résolu par ton `solve`, à 1e−9 près de NumPy, en moins d'une seconde 🏆 ★★★ ⏱️ 30 min
**Objectif :** rendre ta fonction `solve` à la fois précise et rapide sur un grand système, en vectorisant l'élimination.  
**Prérequis :** fiche §6 ; Ex M09a.27 et M09a.32 · **Parcours :** C

Ta fonction `solve` (M09a.27) passe ses tests sur de petits systèmes. **L'objectif** : qu'elle résolve trois systèmes $400 \times 400$ (deux tirés au hasard, un troisième dont la diagonale est minuscule) à $10^{-9}$ près de `np.linalg.solve`, en **moins d'une seconde** chacun, sans `np.linalg` ni `@` (la règle de `matrices.py`).

Une élimination qui traite les nombres un par un (une boucle Python sur les lignes sous le pivot, puis une sur les colonnes) fait de l'ordre de $\frac{2}{3}n^3 \approx 4 \times 10^7$ opérations Python pour $n = 400$ : plusieurs secondes. Pistes : traiter chaque ligne d'un coup (une opération NumPy sur une ligne entière), ou mieux toutes les lignes sous le pivot d'un coup (une opération sur le bloc `M[k + 1:]`, avec le broadcasting d'une colonne de multiplicateurs), et garder le pivot partiel. La vérification s'arrête au premier système trop lent.

La vérification essaie ta fonction sur les trois systèmes ; une dernière cellule enregistre si l'objectif est atteint sur les trois.

Dans ta copie : combien de temps prenait ta première version, et combien la version finale ? Quelle opération as-tu vectorisée ? Pourquoi le troisième système échoue-t-il sans pivot partiel ?

In [ ]:
with checked("M09a.36 défi"):
    problems, worst_error, slowest, ok_36 = [], 0.0, 0.0, False
    for seed_t, kind_t in [(36, "random"), (37, "random"), (38, "needs-pivoting")]:
        rng_t = np.random.default_rng(seed_t)
        A_t = rng_t.normal(size=(400, 400))
        if kind_t == "needs-pivoting":
            A_t[np.arange(400), np.arange(400)] = 1e-14          # tiny diagonal: elimination without swaps fails
        b_t = rng_t.normal(size=400)
        exact_t = np.linalg.solve(A_t, b_t)
        start_t = time.perf_counter()
        try:
            x_t = mymath.matrices.solve(A_t.copy(), b_t.copy())
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"solve lève {type(exc).__name__} ({exc})")
            break
        seconds_t = time.perf_counter() - start_t
        slowest = max(slowest, seconds_t)
        x_t = np.asarray(x_t, dtype=float)
        if x_t.shape != (400,):
            problems.append(f"solve rend la forme {x_t.shape}, pas (400,)")
            break
        worst_error = max(worst_error, float(np.max(np.abs(x_t - exact_t))))
        if seconds_t > 1:
            break                                                # too slow: the other systems would be too
    if not problems:
        print(f"plus grande erreur : {worst_error:.1e} · temps le plus long : {slowest:.3f} s")
        if worst_error > 1e-9:
            problems.append("l'erreur dépasse 1e-9 : vérifie le pivot partiel (le troisième système a une diagonale "
                            "minuscule)")
        if slowest > 1:
            problems.append(f"plus d'une seconde ({slowest:.1f} s) : remplace les boucles sur les éléments par des "
                            "opérations NumPy sur des lignes entières, ou sur tout le bloc M[k + 1:] d'un coup")
    if problems:
        print("❌ Ex M09a.36 : " + " ; ".join(problems) + ".")
    else:
        ok_36 = True
        print("✅ Ex M09a.36 : défi réussi : trois systèmes 400 × 400, à 1e-9 près, en moins d'une seconde chacun.")
with checked("M09a.36"):
    mymath.matrices.solve([[2, 1], [1, 1]], [3, 2])          # ⏳ while solve is not written
    wbm.check("M09a.36", bool(ok_36), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu prévoir la forme d'un produit, et calculer un produit matrice-vecteur sous ses deux lectures ?
2. Sais-tu calculer un produit matriciel, son coût, et écrire une couche $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$ ?
3. Sais-tu appliquer les règles de la transposée, et calculer $\mathbf{X}^\top\mathbf{X}$ ?
4. Sais-tu résoudre un système par le pivot de Gauss, et décrire toutes ses solutions ?
5. Sais-tu calculer une inverse et un rang, et dire quand une matrice est singulière ?

**La suite** : le chapitre 9 de workbookIA (overfitting et régularisation, avec des régressions écrites en produits matriciels), puis M09b, les moindres carrés.